In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))

import lora_transfer_pruning
from  lora_transfer_pruning.core.pruning_instrumentor import PruningInstrumentor
from transformers import AutoModelForCausalLM
import torch

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [2]:
import importlib
# importlib.reload(PruningInstrumentor)

In [3]:
MODEL = "meta-llama/Llama-3.1-8B-Instruct" 
DEVICE = "cuda:0"
def load_model(device=DEVICE):
    model = AutoModelForCausalLM.from_pretrained(
        MODEL,
        #quantization_config=quantization_config,
        #dtype=torch.bfloat16,
        device_map=device,
        # cache_dir="/glazkov-dev/.cache",
    )
    return model

In [4]:
model = load_model()

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [5]:
from transformer_lens.model_bridge import TransformerBridge
import transformer_lens

bridge = TransformerBridge.boot_transformers(
    MODEL,
    hf_model=model,
    dtype=torch.float16,
)

In [6]:
bridge


TransformerBridge(
  (embed): EmbeddingBridge(
    (hook_in): HookPoint(name='embed.hook_in')
    (hook_out): HookPoint(name='embed.hook_out')
    (_original_component): Embedding(128256, 4096)
  )
  (rotary_emb): RotaryEmbeddingBridge(
    (hook_in): HookPoint(name='rotary_emb.hook_in')
    (hook_out): HookPoint(name='rotary_emb.hook_out')
    (hook_cos): HookPoint(name='rotary_emb.hook_cos')
    (hook_sin): HookPoint(name='rotary_emb.hook_sin')
    (_original_component): LlamaRotaryEmbedding()
  )
  (blocks): ModuleList(
    (0): BlockBridge(
      (hook_in): HookPoint(name='blocks.0.hook_in')
      (hook_out): HookPoint(name='blocks.0.hook_out')
      (hook_mlp_in): HookPoint(name='blocks.0.hook_mlp_in')
      (_original_component): LlamaDecoderLayer(
        (self_attn): PositionEmbeddingsAttentionBridge(
          (hook_in): HookPoint(name='blocks.0.attn.hook_in')
          (hook_out): HookPoint(name='blocks.0.attn.hook_out')
          (hook_attn_scores): HookPoint(name='blocks.0.

In [7]:
from datasets import load_dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL)
validation_dataset = load_dataset(
    "Salesforce/wikitext",
    "wikitext-2-raw-v1",
    split="validation",
)
validation_dataset

Dataset({
    features: ['text'],
    num_rows: 3760
})

In [8]:
for i, text in enumerate(validation_dataset):
    print(f"{i}: {text}")
    if i > 10:
        break

0: {'text': ''}
1: {'text': ' = Homarus gammarus = \n'}
2: {'text': ''}
3: {'text': ' Homarus gammarus , known as the European lobster or common lobster , is a species of clawed lobster from the eastern Atlantic Ocean , Mediterranean Sea and parts of the Black Sea . It is closely related to the American lobster , H. americanus . It may grow to a length of 60 cm ( 24 in ) and a mass of 6 kilograms ( 13 lb ) , and bears a conspicuous pair of claws . In life , the lobsters are blue , only becoming " lobster red " on cooking . Mating occurs in the summer , producing eggs which are carried by the females for up to a year before hatching into planktonic larvae . Homarus gammarus is a highly esteemed food , and is widely caught using lobster pots , mostly around the British Isles . \n'}
4: {'text': ''}
5: {'text': ' = = Description = = \n'}
6: {'text': ''}
7: {'text': ' Homarus gammarus is a large crustacean , with a body length up to 60 centimetres ( 24 in ) and weighing up to 5 – 6 kilogram

In [9]:
CONTEXT_LENGTH = 256
NUM_EVAL_BLOCKS = 32 #(block=batch)
EVAL_BATCH_SIZE = 1

validation_text = "\n\n".join(
    text for text in validation_dataset["text"] if text.strip()
)
validation_tokens = tokenizer(
    validation_text,
    add_special_tokens=False,
    return_tensors="pt",
).input_ids[0]

num_blocks = NUM_EVAL_BLOCKS
assert num_blocks > 0, "Validation split does not contain enough tokens"
evaluation_blocks = validation_tokens[: num_blocks * CONTEXT_LENGTH].reshape(
    num_blocks, CONTEXT_LENGTH
)
evaluation_blocks.shape

torch.Size([32, 256])

In [10]:
ignored_params = []
for name, param in model.named_parameters():
    if "norm" in name:
        ignored_params.append(param)

In [11]:
import torch
import torch.nn as nn
import torch_pruning as tp

example_inputs = evaluation_blocks[:4].to(DEVICE)

def trace_forward(model, input_ids):
    return model(
        input=input_ids,
        use_cache=False,
        return_type="logits",
        #return_dict=True,
    ) #for compatability with tp

DG = tp.DependencyGraph().build_dependency(
    bridge,
    example_inputs=example_inputs,
    forward_fn=trace_forward,
    ignored_params=ignored_params,
    #unwrapped_parameters=list(zip(ignored_params, [_] * len(ignored_params)))
)


/glazkov-dev/LoRa-Transfer-Pruning/.venv/lib/python3.10/site-packages/torch_pruning/dependency/graph.py:390: UserWarning: Unwrapped parameters detected: ['model.layers.31._original_component.post_attention_layernorm._original_component.weight', 'model.layers.3._original_component.post_attention_layernorm._original_component.weight', 'model.layers.13._original_component.input_layernorm._original_component.weight', 'model.layers.16._original_component.input_layernorm._original_component.weight', 'model.layers.18._original_component.input_layernorm._original_component.weight', 'model.layers.18._original_component.post_attention_layernorm._original_component.weight', 'model.layers.29._original_component.input_layernorm._original_component.weight', 'model.layers.30._original_component.input_layernorm._original_component.weight', 'model.layers.30._original_component.post_attention_layernorm._original_component.weight', 'model.layers.0._original_component.post_attention_layernorm._original_co

In [12]:
bridge.blocks[0].attn.q

LinearBridge(4096 -> 4096, bias=False, original_component=Linear)

In [13]:
bridge.get_submodule("blocks.0.attn.q.hook_out")

HookPoint(name='blocks.0.attn.q.hook_out')

In [14]:
#name of module, cols (in), rows(out)


#local configuration
d = {"blocks.0.attn.q": (None, [2, 6, 9]), #repeat indices for qkvo
     "blocks.0.mlp.up_proj": (None, [1, 3, 5])}

In [15]:
"blocks.0.attn.q"[-2:]

'.q'

In [16]:
group = DG.get_pruning_group(
    bridge.blocks[0].attn.q._original_component, 
    tp.prune_linear_out_channels, 
    idxs=[2, 6, 9] )

In [17]:
def manually_indices_repeating(num_heads: int, head_dim: int, pruning_indices: torch.Tensor):
    all_indices = []
    for head_num in range(num_heads):
        all_indices.append(
            pruning_indices+head_num*head_dim)
    return torch.cat(all_indices)

In [18]:
repeated_idxs = manually_indices_repeating(
    bridge.model.config.num_attention_heads,
    bridge.model.config.head_dim,
    torch.tensor([2, 6, 9])
)
repeated_idxs

tensor([   2,    6,    9,  130,  134,  137,  258,  262,  265,  386,  390,  393,
         514,  518,  521,  642,  646,  649,  770,  774,  777,  898,  902,  905,
        1026, 1030, 1033, 1154, 1158, 1161, 1282, 1286, 1289, 1410, 1414, 1417,
        1538, 1542, 1545, 1666, 1670, 1673, 1794, 1798, 1801, 1922, 1926, 1929,
        2050, 2054, 2057, 2178, 2182, 2185, 2306, 2310, 2313, 2434, 2438, 2441,
        2562, 2566, 2569, 2690, 2694, 2697, 2818, 2822, 2825, 2946, 2950, 2953,
        3074, 3078, 3081, 3202, 3206, 3209, 3330, 3334, 3337, 3458, 3462, 3465,
        3586, 3590, 3593, 3714, 3718, 3721, 3842, 3846, 3849, 3970, 3974, 3977])

In [19]:
repeated_idxs.shape

torch.Size([96])

In [20]:
group = DG.get_pruning_group(
    bridge.blocks[0].attn.q._original_component, 
    tp.prune_linear_out_channels, 
    idxs=repeated_idxs )

In [21]:
torch.as_tensor([1, 2, 3])

tensor([1, 2, 3])

In [22]:
bridge.blocks[0].attn.config.head_dim

128

In [23]:
bridge.blocks[0].attn.config.n_key_value_heads

8

In [24]:
bridge.config.head_dim

128

In [25]:
print(group)


--------------------------------
          Pruning Group
--------------------------------
[0] prune_out_channels on blocks.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=4096, out_features=4096, bias=False)) => prune_out_channels on blocks.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=4096, out_features=4096, bias=False)), len(idxs)=96
[1] prune_out_channels on blocks.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=4096, out_features=4096, bias=False)) => prune_out_channels on _Reshape_1521(), len(idxs)=96
[2] prune_out_channels on _Reshape_1521() => prune_out_channels on _ElementWiseOp_1520(TransposeBackward0), len(idxs)=96
[3] prune_out_channels on _ElementWiseOp_1520(TransposeBackward0) => prune_out_channels on _Slice_1519(), len(idxs)=96
[4] prune_out_channels on _ElementWiseOp_1520(TransposeBackward0) => prune_out_channels on 

In [26]:
for i, (dep, idx) in enumerate(group):
    print(dep.target.name)

blocks.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=4096, out_features=4096, bias=False))
_Reshape_1521()
_ElementWiseOp_1520(TransposeBackward0)
_Slice_1519()
_Slice_1525()
_ElementWiseOp_1515(MulBackward0)
_ElementWiseOp_1514(AddBackward0)
_ElementWiseOp_1516(MulBackward0)
_ExpandOp_1513()
_ElementWiseOp_1512(CloneBackward0)
_Reshape_1492()
_ElementWiseOp_1491(BmmBackward0)
_Reshape_1493()
_Reshape_1490()
_ElementWiseOp_1489(MulBackward0)
_ElementWiseOp_1488(MaskedFillBackward0)
_ElementWiseOp_1487(ToCopyBackward0)
_ElementWiseOp_1486(SoftmaxBackward0)
_ElementWiseOp_1485(ToCopyBackward0)
_ExpandOp_1484()
_Reshape_1464()
_ElementWiseOp_1463(BmmBackward0)
_Reshape_1465()
_Reshape_1462()
_ElementWiseOp_1461(TransposeBackward0)
_ElementWiseOp_1460(CloneBackward0)
_Reshape_1459()
_Reshape_1457()
_ElementWiseOp_1456(MmBackward0)
_ElementWiseOp_1458(TBackward0)
blocks.0._original_component.self_attn._original_component.o_proj._original_

In [27]:
name = "blocks.0._original_component.self_attn._original_component.o_proj._original_component (Linear(in_features=4096, out_features=4096, bias=False))" 
#its full name!

In [28]:
".".join(name.split(".")[:-1])

'blocks.0._original_component.self_attn._original_component.o_proj'

In [29]:
name[:name.find(" ")]

'blocks.0._original_component.self_attn._original_component.o_proj._original_component'

In [30]:
bridge.get_submodule("blocks.0._original_component.self_attn._original_component.k_proj")

LinearBridge(4096 -> 1024, bias=False, original_component=Linear)

In [31]:
repeated_idxs_kv = manually_indices_repeating(
    bridge.model.config.num_key_value_heads,
    bridge.model.config.head_dim,
    torch.tensor([2, 6, 9])
)
repeated_idxs_kv

tensor([  2,   6,   9, 130, 134, 137, 258, 262, 265, 386, 390, 393, 514, 518,
        521, 642, 646, 649, 770, 774, 777, 898, 902, 905])

In [32]:
group_kv = DG.get_pruning_group( 
    bridge.blocks[0].mlp.up_proj._original_component, 
    tp.prune_linear_out_channels, 
    idxs=[ 2 ] )

In [33]:
type(bridge.blocks[0].mlp) #goes to original coponent implementation

transformer_lens.model_bridge.generalized_components.gated_mlp.GatedMLPBridge

In [34]:
print(group_kv)


--------------------------------
          Pruning Group
--------------------------------
[0] prune_out_channels on blocks.0._original_component.mlp._original_component.up_proj._original_component (Linear(in_features=4096, out_features=14336, bias=False)) => prune_out_channels on blocks.0._original_component.mlp._original_component.up_proj._original_component (Linear(in_features=4096, out_features=14336, bias=False)), len(idxs)=1
[1] prune_out_channels on blocks.0._original_component.mlp._original_component.up_proj._original_component (Linear(in_features=4096, out_features=14336, bias=False)) => prune_out_channels on _ElementWiseOp_1443(MulBackward0), len(idxs)=1
[2] prune_out_channels on _ElementWiseOp_1443(MulBackward0) => prune_out_channels on _ElementWiseOp_1444(SiluBackward0), len(idxs)=1
[3] prune_out_channels on _ElementWiseOp_1443(MulBackward0) => prune_out_channels on _Reshape_1441(), len(idxs)=1
[4] prune_out_channels on _Reshape_1441() => prune_out_channels on _ElementWiseO

WARN: k.out doesnt link to q.out

But q.out - link to all modules.

So we need use only attn.q for prune in attn and mlp.up_proj for mlp pruning. Also for local pruning use tp.prune_out_channels, because it doesnt touch residual stream.

### Fraction-based pruning comparison

Restart the kernel and run the model/dataset setup cells, but skip the preceding explicit-index pruning cell. This experiment samples channel indices from fractions, records the actual independently sampled RoPE coordinates for each attention layer, then compares activation and structural pruning using the same groups.

In [35]:
# Fraction-based version of the activation-vs-structural comparison.
# Run on a freshly loaded, unpruned bridge; skip the preceding explicit-index
# comparison cell after restarting the kernel.
from compare_utils import compare_tp_and_transfer_pruning

FRACTION_ATTN_LAYERS = [0, 8, 16]
FRACTION_MLP_LAYERS = [4, 12, 20]
ATTN_OUT_FRACTION = 0.1 #[1, 2]
MLP_OUT_FRACTION = 0.3 #[1, 2]
FRACTION_SEED = 0

compare_tp_and_transfer_pruning(bridge,
                                FRACTION_ATTN_LAYERS,
                                FRACTION_MLP_LAYERS,
                                ATTN_OUT_FRACTION,
                                MLP_OUT_FRACTION,
                                FRACTION_SEED,
                                evaluation_blocks,
                                EVAL_BATCH_SIZE
                                )

fraction prune_task:
  blocks.0.attn.q: (None, 0.1)
  blocks.8.attn.q: (None, 0.1)
  blocks.16.attn.q: (None, 0.1)
  blocks.4.mlp.up_proj: (None, 0.3)
  blocks.12.mlp.up_proj: (None, 0.3)
  blocks.20.mlp.up_proj: (None, 0.3)
removed root indices by module:
  blocks.0.attn.q: count=384, idxs=[1, 5, 7, 44, 49, 59, 65, 69, 71, 108, 113, 123, 129, 133, 135, 172, 177, 187, 193, 197, 199, 236, 241, 251, 257, 261, 263, 300, 305, 315, 321, 325, 327, 364, 369, 379, 385, 389, 391, 428, 433, 443, 449, 453, 455, 492, 497, 507, 513, 517, 519, 556, 561, 571, 577, 581, 583, 620, 625, 635, 641, 645, 647, 684, 689, 699, 705, 709, 711, 748, 753, 763, 769, 773, 775, 812, 817, 827, 833, 837, 839, 876, 881, 891, 897, 901, 903, 940, 945, 955, 961, 965, 967, 1004, 1009, 1019, 1025, 1029, 1031, 1068, 1073, 1083, 1089, 1093, 1095, 1132, 1137, 1147, 1153, 1157, 1159, 1196, 1201, 1211, 1217, 1221, 1223, 1260, 1265, 1275, 1281, 1285, 1287, 1324, 1329, 1339, 1345, 1349, 1351, 1388, 1393, 1403, 1409, 1413, 1415, 14

{'baseline': {'loss': 2.26513671875, 'perplexity': 9.632441520690918},
 'transfer_activation': {'loss': 2.430908203125,
  'perplexity': 11.369202613830566},
 'torch_pruning_structural': {'loss': 2.409912109375,
  'perplexity': 11.13298225402832},
 'structural_minus_transfer': {'loss': -0.02099609375,
  'perplexity': -0.2362203598022461}}

So we see a little divergence in more fraction sizes. But it very close.